In [1]:
import numpy as numpy
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from concurrent.futures import ThreadPoolExecutor

In [2]:
# NIFTY50
historical_url_1 = 'https://en.wikipedia.org/w/index.php?title=NIFTY_50#Constituents'

tables = pd.read_html(
    historical_url_1,
    match='Symbol', 
    storage_options={'User-Agent': 'Mozilla/5.0'}
)

nifty50_df_1 = tables[0]
ticker_list_1 = nifty50_df_1['Symbol'].astype(str) + '.NS'

In [3]:
def fetch_mcap(ticker):
    """Fetches the market cap for a single ticker."""
    try:
        # .fast_info is a lightweight dictionary that loads instantly
        # compared to the heavy .info dictionary
        stock = yf.Ticker(ticker)
        mcap = stock.fast_info.market_cap
        return ticker, mcap
    except Exception as e:
        # If a ticker is delisted or throws an error, return None safely
        return ticker, None

In [4]:
print(f"Fetching Market Caps for {len(ticker_list_1)} stocks...")

results = []

# ThreadPoolExecutor opens 10 parallel connections to Yahoo Finance
# This turns a 10-minute download into a 30-second download
with ThreadPoolExecutor(max_workers=10) as executor:
    # executor.map runs the fetch_mcap function on every ticker in the list simultaneously
    for result in executor.map(fetch_mcap, ticker_list_1):
        results.append(result)

# Convert the results into a clean Pandas DataFrame
mcap_df = pd.DataFrame(results, columns=["Ticker", "Market_Cap_Raw"])

# Clean the data: Drop any failures and sort from largest to smallest
mcap_df = mcap_df.dropna().sort_values(by="Market_Cap_Raw", ascending=False).reset_index(drop=True)

mcap_df['Market_Cap_Cr'] = (mcap_df['Market_Cap_Raw'] / 10000000).round(2)
mcap_df['Market_Cap_pct'] = 100 * (mcap_df['Market_Cap_Cr'] / mcap_df['Market_Cap_Cr'].sum())
mcap_df = mcap_df.drop(columns=['Market_Cap_Raw'])
mcap_df['Cumulative_Cap_pct'] = mcap_df['Market_Cap_pct'].cumsum()
nifty50_mcap = mcap_df.copy()
print("\n--- Top Constituents by Market Cap ---")
display(nifty50_mcap)

Fetching Market Caps for 50 stocks...

--- Top Constituents by Market Cap ---


,Ticker,Market_Cap_Cr,Market_Cap_pct,Cumulative_Cap_pct
0,RELIANCE.NS,1827154.39,9.422983,9.422983
1,HDFCBANK.NS,1247315.90,6.432646,15.855628
2,BHARTIARTL.NS,1139105.19,5.874582,21.730211
3,SBIN.NS,984629.93,5.077924,26.808135
4,ICICIBANK.NS,946638.84,4.881997,31.690131
5,TCS.NS,913313.85,4.710133,36.400265
6,BAJFINANCE.NS,574732.22,2.964003,39.364268
7,LT.NS,544736.58,2.809310,42.173578
8,INFY.NS,523028.08,2.697355,44.870934
9,HINDUNILVR.NS,506480.97,2.612019,47.482953


In [5]:
historical_url_2 = 'https://en.wikipedia.org/w/index.php?title=NIFTY_500#Constituents' # 26 Nov 2024

tables = pd.read_html(
    historical_url_2,
    match='Symbol', 
    storage_options={'User-Agent': 'Mozilla/5.0'}
)

nifty500_df = tables[0]
header = nifty500_df.iloc[0]
nifty500_df = nifty500_df[1:]
nifty500_df.columns = header
nifty500_df = nifty500_df.reset_index(drop=True)
ticker_list_2 = nifty500_df['Symbol'].astype(str) + '.NS'

print("Historical Constituents List:")
print(ticker_list_2.tolist())

Historical Constituents List:
['360ONE.NS', '3MINDIA.NS', 'ABB.NS', 'ACC.NS', 'AIAENG.NS', 'APLAPOLLO.NS', 'AUBANK.NS', 'AARTIIND.NS', 'AAVAS.NS', 'ABBOTINDIA.NS', 'ACE.NS', 'ADANIENSOL.NS', 'ADANIENT.NS', 'ADANIGREEN.NS', 'ADANIPORTS.NS', 'ADANIPOWER.NS', 'ATGL.NS', 'AWL.NS', 'ABCAPITAL.NS', 'ABFRL.NS', 'AEGISLOG.NS', 'AETHER.NS', 'AFFLE.NS', 'AJANTPHARM.NS', 'APLLTD.NS', 'ALKEM.NS', 'ALKYLAMINE.NS', 'ALLCARGO.NS', 'ALOKINDS.NS', 'ARE&M.NS', 'AMBER.NS', 'AMBUJACEM.NS', 'ANANDRATHI.NS', 'ANGELONE.NS', 'ANURAS.NS', 'APARINDS.NS', 'APOLLOHOSP.NS', 'APOLLOTYRE.NS', 'APTUS.NS', 'ACI.NS', 'ASAHIINDIA.NS', 'ASHOKLEY.NS', 'ASIANPAINT.NS', 'ASTERDM.NS', 'ASTRAZEN.NS', 'ASTRAL.NS', 'ATUL.NS', 'AUROPHARMA.NS', 'AVANTIFEED.NS', 'DMART.NS', 'AXISBANK.NS', 'BEML.NS', 'BLS.NS', 'BSE.NS', 'BAJAJ-AUTO.NS', 'BAJFINANCE.NS', 'BAJAJFINSV.NS', 'BAJAJHLDNG.NS', 'BALAMINES.NS', 'BALKRISIND.NS', 'BALRAMCHIN.NS', 'BANDHANBNK.NS', 'BANKBARODA.NS', 'BANKINDIA.NS', 'MAHABANK.NS', 'BATAINDIA.NS', 'BAYERCROP.NS', 

In [6]:
print(f"Fetching Market Caps for {len(ticker_list_2)} stocks...")

results = []

# ThreadPoolExecutor opens 10 parallel connections to Yahoo Finance
# This turns a 10-minute download into a 30-second download
with ThreadPoolExecutor(max_workers=10) as executor:
    # executor.map runs the fetch_mcap function on every ticker in the list simultaneously
    for result in executor.map(fetch_mcap, ticker_list_2):
        results.append(result)

# Convert the results into a clean Pandas DataFrame
mcap_df = pd.DataFrame(results, columns=["Ticker", "Market_Cap_Raw"])

# Clean the data: Drop any failures and sort from largest to smallest
mcap_df = mcap_df.dropna().sort_values(by="Market_Cap_Raw", ascending=False).reset_index(drop=True)

mcap_df['Market_Cap_Cr'] = (mcap_df['Market_Cap_Raw'] / 10000000).round(2)
mcap_df['Market_Cap_pct'] = 100 * (mcap_df['Market_Cap_Cr'] / mcap_df['Market_Cap_Cr'].sum())
mcap_df = mcap_df.drop(columns=['Market_Cap_Raw'])
mcap_df['Cumulative_Cap_pct'] = mcap_df['Market_Cap_pct'].cumsum()
nifty500_mcap = mcap_df.copy()

print("\n--- Top Constituents by Market Cap ---")
display(mcap_df)

Fetching Market Caps for 500 stocks...


ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: GMRINFRASTRUCT.NS"}}}
ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: ISEC.NS"}}}
ERROR:yfinance:$ISEC.NS: possibly delisted; no price data found  (period=1y) (Yahoo error = "No data found, symbol may be delisted")
ERROR:yfinance:$ISEC.NS: possibly delisted; no price data found  (period=5d) (Yahoo error = "No data found, symbol may be delisted")
ERROR:yfinance:$LTIM.NS: possibly delisted; no price data found  (period=5d)
ERROR:yfinance:$PEL.NS: possibly delisted; no price data found  (period=1y) (Yahoo error = "No data found, symbol may be delisted")
ERROR:yfinance:$PEL.NS: possibly delisted; no price data found  (period=5d) (Yahoo error = "No data found, symbol may be delisted")
ERROR:yfinance:$SUVENPHAR.NS: possibly delisted; no price data found  (period=1y) (Yahoo erro


--- Top Constituents by Market Cap ---


,Ticker,Market_Cap_Cr,Market_Cap_pct,Cumulative_Cap_pct
0,RELIANCE.NS,1827154.39,4.803672,4.803672
1,HDFCBANK.NS,1247315.90,3.279250,8.082921
2,BHARTIARTL.NS,1139105.19,2.994759,11.077680
3,SBIN.NS,984629.93,2.588637,13.666317
4,ICICIBANK.NS,946638.84,2.488756,16.155073
...,...,...,...,...
489,EASEMYTRIP.NS,2913.12,0.007659,99.975274
490,QUESS.NS,2885.08,0.007585,99.982859
491,PRINCEPIPE.NS,2651.98,0.006972,99.989831
492,RAYMOND.NS,2570.88,0.006759,99.996590


In [7]:
pd.set_option('display.max_rows', None)
mcap_df

,Ticker,Market_Cap_Cr,Market_Cap_pct,Cumulative_Cap_pct
0,RELIANCE.NS,1827154.39,4.803672,4.803672
1,HDFCBANK.NS,1247315.90,3.279250,8.082921
2,BHARTIARTL.NS,1139105.19,2.994759,11.077680
3,SBIN.NS,984629.93,2.588637,13.666317
4,ICICIBANK.NS,946638.84,2.488756,16.155073
5,TCS.NS,913313.85,2.401143,18.556217
6,BAJFINANCE.NS,574732.22,1.510997,20.067214
7,LT.NS,544736.58,1.432137,21.499351
8,INFY.NS,523028.08,1.375064,22.874415
9,HINDUNILVR.NS,506480.97,1.331561,24.205977
